# EvalForge — reproducible GPU experiment
Enable internet and a CUDA GPU. This notebook creates a Python 3.11 environment independently of Kaggle’s host Python. The data mirror is pinned and checksum verified; model and dataset terms apply. No secret tokens are required for the public model and data. Outputs contain the adapter, manifest, and paired evaluation report.


In [1]:
import subprocess, sys, os
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'venv', '--python', '3.11', '--seed', '/kaggle/temp/evalforge-env'], check=True)
repo = Path('/kaggle/working/EvalForge')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Navaneeth-007/EvalForge.git', str(repo)], check=True)
print(subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True))
os.environ['HF_HOME'] = '/kaggle/temp/evalforge-model-cache'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['WANDB_MODE'] = 'disabled'
subprocess.run(['uv', 'pip', 'install', '--python', '/kaggle/temp/evalforge-env/bin/python', '-r', str(repo / 'requirements-ml.txt')], check=True)


Using CPython 3.11.17
Creating virtual environment with seed packages at: /kaggle/temp/evalforge-env
 + packaging==26.3
 + pip==26.2.2
 + setuptools==84.2.1
 + wheel==0.48.0
Activate with: source /kaggle/temp/evalforge-env/bin/activate
Cloning into '/kaggle/working/EvalForge'...


38c7410fbcd727403b7c062377924c6c9bb35bc4



Using Python 3.11.17 environment at: /kaggle/temp/evalforge-env
Resolved 116 packages in 1.16s
Prepared 115 packages in 38.49s
Uninstalled 1 package in 0.94ms
Installed 115 packages in 367ms
 + accelerate==1.1.1
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + altair==5.5.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anyio==4.15.1
 + attrs==26.1.0
 + bert-score==0.3.13
 + bitsandbytes==0.44.1
 + blinker==1.9.0
 + cachetools==5.5.2
 + certifi==2026.10.23
 + charset-normalizer==3.5.2
 + click==8.5.0
 + colorama==0.4.6
 + contourpy==1.3.3
 + cycler==0.12.1
 + datasets==3.1.0
 + dill==0.3.8
 + docker-pycreds==0.4.0
 + fastapi==0.162.0
 + filelock==4.2.4
 + fonttools==4.68.0
 + frozenlist==1.8.0
 + fsspec==2024.9.0
 + gitdb==4.0.12
 + gitpython==3.2.1
 + h11==0.16.0
 + httptools==0.9.0
 + huggingface-hub==0.26.5
 + idna==3.20
 + jinja2==3.1.6
 + jsonschema==4.26.0
 + jsonschema-specifications==2025.9.1
 + kiwisolver==1.5.1
 + lxml==6.1.3
 + markdown-it-py==4.2.0

CompletedProcess(args=['uv', 'pip', 'install', '--python', '/kaggle/temp/evalforge-env/bin/python', '-r', '/kaggle/working/EvalForge/requirements-ml.txt'], returncode=0)

## Train and evaluate
The default run reproduces the published pilot: 1,000 selected training rows, one configured epoch, and paired evaluation on 200 development examples. Set RUN_ARGS=[] for the full three-epoch experiment. Batch size is 1 with gradient accumulation 16 on one GPU. Runtime depends on the assigned hardware. Keep the session active and preserve the output artifacts.


In [3]:
RUN_ARGS = ['--epochs', '2', '--eval-n', '200']
subprocess.run(['/kaggle/temp/evalforge-env/bin/python', '-u', '-m', 'scripts.run_gpu', *RUN_ARGS], cwd=str(repo), env=os.environ.copy(), check=True)


INFO:datasets:PyTorch version 2.5.1 available.
INFO:train.dataset_prep:Loading Spider dataset from Hugging Face...
Generating validation split: 100%|██████████| 1034/1034 [00:00<00:00, 182338.04 examples/s]
INFO:train.dataset_prep:Train: 7000 examples | Validation: 1034 examples
INFO:train.dataset_prep:Formatting prompts...
Formatting: 100%|██████████| 1034/1034 [00:00<00:00, 13714.16 examples/s]
INFO:train.dataset_prep:Loading tokenizer: microsoft/Phi-3-mini-4k-instruct
Saving the dataset (1/1 shards): 100%|██████████| 1034/1034 [00:00<00:00, 70491.36 examples/s]
INFO:train.dataset_prep:Dataset saved to ./data/processed
INFO:train.dataset_prep:Saved 200 eval samples to data/processed/eval_samples.json
INFO:train.train:Loading pre-processed dataset from ./data/processed
INFO:train.train:Train: 6918 | Eval: 1034
INFO:train.train:Loading model: microsoft/Phi-3-mini-4k-instruct
INFO:accelerate.utils.modeling:We will use 90% of the memory on device 0 for storing the model, and 10% for the 

trainable params: 25,165,824 || all params: 3,846,245,376 || trainable%: 0.6543


INFO:train.train:Starting training...
  0%|          | 0/864 [00:00<?, ?it/s]/kaggle/temp/evalforge-env/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
  1%|          | 10/864 [07:50<10:06:07, 42.58s/it]

{'loss': 0.8281, 'grad_norm': 0.5919142365455627, 'learning_rate': 7.692307692307693e-05, 'epoch': 0.02}


  2%|▏         | 20/864 [13:27<7:22:09, 31.43s/it] 

...

{'train_runtime': 29611.8083, 'train_samples_per_second': 0.467, 'train_steps_per_second': 0.029, 'train_loss': 0.1106294470804709, 'epoch': 2.0}


100%|██████████| 864/864 [8:13:31<00:00, 34.27s/it]
INFO:train.train:Saving adapter to ./outputs/phi3-sql-qlora/final
INFO:train.train:Training complete.
INFO:eval.run_eval:Loading eval samples from data/processed/eval_samples.json
INFO:eval.run_eval:Evaluating on 200 examples
INFO:eval.run_eval:
INFO:eval.run_eval:EVALUATING BASE MODEL (before fine-tuning)
INFO:eval.run_eval:============================================================
INFO:eval.run_eval:Loading model: microsoft/Phi-3-mini-4k-instruct
INFO:accelerate.utils.modeling:We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).
Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.95s/it]
INFO:eval.run_eval:  Generated 4/200
INFO:eval.run_eval:  Generated 8/200
INFO:eval.run_eval:  Generated 12/200
INFO:eval.run_eval:  Generated 16/200
INFO:eval.run_eval:  Generated 20/200
INFO:eval.run_


EVALFORGE — BEFORE vs AFTER RESULTS
Metric                         Base Model    Fine-tuned           Δ
─────────────────────────────────────────────────────────────────
Exact Match Accuracy               0.1450        0.2500   (+0.1050)
Execution Accuracy                 0.5000        0.5600   (+0.0600)
BLEU-4                             0.3076        0.5246   (+0.2170)
BERTScore F1                          N/A           N/A            
─────────────────────────────────────────────────────────────────

📋 Copy the above table into your README.md

Experiment complete: preserve outputs/ and update the model card with measured evidence.


INFO:eval.run_eval:
Full results saved to outputs/eval_results.json


CompletedProcess(args=['/kaggle/temp/evalforge-env/bin/python', '-u', '-m', 'scripts.run_gpu', '--epochs', '2', '--eval-n', '200'], returncode=0)

## Inspect and save evidence
These are development metrics, not a blind test score. Inspect incorrect predictions before claiming an improvement. Do not publish the model card until its pending fields have been replaced with measured values.


In [4]:
import json, shutil
manifest = json.loads((repo / 'outputs/phi3-sql-qlora/training_manifest.json').read_text())
report = json.loads((repo / 'outputs/eval_results.json').read_text())
print('Training examples:', manifest['train_examples'])
print('GPU:', manifest['gpu'], 'Elapsed seconds:', manifest['elapsed_seconds'])
print('Base metrics:', report['base_model_metrics'])
print('Adapter metrics:', report['finetuned_model_metrics'])
shutil.copy2(
    repo / 'data/processed/eval_samples.json',
    repo / 'outputs/eval_samples.json',
)

shutil.make_archive(
    '/kaggle/working/evalforge-evidence',
    'zip',
    repo / 'outputs',
)

Training examples: 6918
GPU: Tesla T4 Elapsed seconds: 29695.994768554
Base metrics: {'exact_match': 0.145, 'exact_match_details': {'exact_match_accuracy': 0.145, 'num_correct': 29, 'num_total': 200, 'per_example': [{'prediction': 'SELECT COUNT(*) FROM singer;', 'gold': 'SELECT count(*) FROM singer', 'prediction_normalised': 'select count(*) from singer', 'gold_normalised': 'select count(*) from singer', 'match': True}, {'prediction': 'SELECT COUNT(*) FROM singer;', 'gold': 'SELECT count(*) FROM singer', 'prediction_normalised': 'select count(*) from singer', 'gold_normalised': 'select count(*) from singer', 'match': True}, {'prediction': 'SELECT Name, Country, Age\nFROM singer\nORDER BY Age DESC;', 'gold': 'SELECT name ,  country ,  age FROM singer ORDER BY age DESC', 'prediction_normalised': 'select name, country, age from singer order by age desc', 'gold_normalised': 'select name , country , age from singer order by age desc', 'match': False}, {'prediction': 'SELECT Name, Country, A

'/kaggle/working/evalforge-evidence.zip'